# 01_03_Data Collection SNCB

Unlike the two previous notebooks, this notebook does not ingest data from an open portal data, but **extracts data from a SNCB PDF report** using the `camelot` library.

In [2]:
from pathlib import Path

import camelot
import pandas as pd

In [3]:
DATA_DIR = Path("../data/intermediate/")

The passenger counts from the National Railway Company of Belgium (SNCB/NMBS) are handled differently from the Infrabel and Statbel open datasets. Required to calculate a new passenger-weighted punctuality metric and to enrich the punctuality fact table.  

This file is manually sourced and stored directly in the Git repository for the following reasons : 

* **Lack of a persistent Open Data Portal**: The file is hosted on a commercial website where URLs are subject to change, which would break an automated download pipeline over time.  

* **Data availability and continuity**: The source only provides the most recent annual count (currently, 2024). By committing the PDF to this repo, we ensure the project remains **reproducible** even if the original link disappears next year.  

* **Security barriers**: The SNCB website implements advanced anti-bot measures (dynamic sessions and JavaScript-based tokens) that prevent standard Python scripts from accessing the file reliably without heavy and **fragile workarounds**.

## 1. Extraction from PDF ##

The cell below sets the PDF file path.

In [4]:
pdf_path = Path("../data/raw/voyageurs-montes-2024-def-1.pdf")

In [5]:
tables = camelot.read_pdf(pdf_path, pages="1-10")

## 2. DataFrame Transformation ##

In [6]:
dfs_sncb = [table.df for table in tables]
sncb_passenger_counts = pd.concat(dfs_sncb, ignore_index=True)

headers = sncb_passenger_counts.iloc[0]
sncb_passenger_counts.columns = headers

sncb_passenger_counts = (
    sncb_passenger_counts.drop(sncb_passenger_counts.index[0]).reset_index(drop=True)
)

sncb_passenger_counts.head(10)

,Station \nGare,Gem. instappende tijdens week \nMoy. montés en semaine,Gem. instappende op zaterdag\nMoy. montés le samedi,Gem. instappende op zondag\nMoy. montés le dimanche
0,AALST,5.883,2.220,1.872
1,AALST-KERREBROEK,15,-,-
2,AALTER,2.348,622,620
3,AARSCHOT,5.804,2.243,1.758
4,AARSELE,26,-,-
5,ACREN,253,98,91
6,AISEAU,147,71,73
7,ALKEN,695,109,63
8,AMAY,314,67,76
9,AMPSIN,191,50,42


## 3. Export to Silver

Export as a `.parquet` file

In [ ]:
sncb_passenger_counts.to_parquet(DATA_DIR / "passengers_sncb_2024.parquet")

### 3.1. Export Verification

In [ ]:
df_to_check = pd.read_parquet(DATA_DIR / "passengers_sncb_2024.parquet")
df_to_check.head()

,Station \nGare,Gem. instappende tijdens week \nMoy. montés en semaine,Gem. instappende op zaterdag\nMoy. montés le samedi,Gem. instappende op zondag\nMoy. montés le dimanche
0,AALST,5.883,2.220,1.872
1,AALST-KERREBROEK,15,-,-
2,AALTER,2.348,622,620
3,AARSCHOT,5.804,2.243,1.758
4,AARSELE,26,-,-
